In [1]:
# --- Grand Finale Project: Credit Score Classification ---
# Step 1: Load and Explore the Data

import pandas as pd
import numpy as np
import seaborn as sns

# Load the dataset from the CSV file into a pandas DataFrame.
try:
    df = pd.read_csv('train.csv')
    print("Dataset 'train.csv' loaded successfully!")
except FileNotFoundError:
    print("Error: 'train.csv' not found. Please make sure you have downloaded the file and placed it in the same directory as your notebook.")

if 'df' in locals():
    print("\n--- First 5 Rows ---")
    print(df.head())

    print("\n--- Data Info (Types and Missing Values) ---")
    df.info()

    print("\n--- Target Variable Distribution ---")
    print(df['Credit_Score'].value_counts())

C:\Users\Mayank\AppData\Local\Temp\ipykernel_23732\2678528094.py:12: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


Dataset 'train.csv' loaded successfully!

--- First 5 Rows ---
       ID Customer_ID     Month           Name   Age          SSN Occupation  \
0  0x1602   CUS_0xd40   January  Aaron Maashoh    23  821-00-0265  Scientist   
1  0x1603   CUS_0xd40  February  Aaron Maashoh    23  821-00-0265  Scientist   
2  0x1604   CUS_0xd40     March  Aaron Maashoh  -500  821-00-0265  Scientist   
3  0x1605   CUS_0xd40     April  Aaron Maashoh    23  821-00-0265  Scientist   
4  0x1606   CUS_0xd40       May  Aaron Maashoh    23  821-00-0265  Scientist   

  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  ...  Credit_Mix  \
0      19114.12            1824.843333                  3  ...           _   
1      19114.12                    NaN                  3  ...        Good   
2      19114.12                    NaN                  3  ...        Good   
3      19114.12                    NaN                  3  ...        Good   
4      19114.12            1824.843333                  3  ...    

In [2]:
# --- Step 2: Data Cleaning and Preprocessing ---
# 2.1: Dropping Unnecessary Identifier Columns

print("Original number of columns:", df.shape[1])

columns_to_drop = ['ID', 'Customer_ID', 'Name', 'SSN']
df.drop(columns=columns_to_drop, inplace=True)

print(f"Dropped columns: {columns_to_drop}")
print("New number of columns:", df.shape[1])

print("\n--- Data after dropping identifiers ---")
print(df.head())
df.info()

Original number of columns: 28
Dropped columns: ['ID', 'Customer_ID', 'Name', 'SSN']
New number of columns: 24

--- Data after dropping identifiers ---
      Month   Age Occupation Annual_Income  Monthly_Inhand_Salary  \
0   January    23  Scientist      19114.12            1824.843333   
1  February    23  Scientist      19114.12                    NaN   
2     March  -500  Scientist      19114.12                    NaN   
3     April    23  Scientist      19114.12                    NaN   
4       May    23  Scientist      19114.12            1824.843333   

   Num_Bank_Accounts  Num_Credit_Card  Interest_Rate Num_of_Loan  \
0                  3                4              3           4   
1                  3                4              3           4   
2                  3                4              3           4   
3                  3                4              3           4   
4                  3                4              3           4   

                        

In [3]:
# 2.2: Cleaning 'Dirty' Numerical Columns and Fixing Data Types

print("Cleaning numerical columns that are stored as text...")

# We create a list of columns that we want to convert to a numerical format.

# We will handle 'Credit_History_Age' separately as it's a special case.
cols_to_clean = [
    'Age',
    'Annual_Income',
    'Num_of_Loan',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Outstanding_Debt',
    'Amount_invested_monthly',
    'Monthly_Balance'
]

# This loop goes through each column in our list
for col in cols_to_clean:
    # We use pd.to_numeric. This function is smart and tries to convert things to numbers.
    # The 'errors='coerce'' part is a safety net: if it finds a value it truly can't
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Columns cleaned and converted to numeric types.")

print("\n--- Data Info after cleaning 'dirty' numerics ---")
df.info()

Cleaning numerical columns that are stored as text...
Columns cleaned and converted to numeric types.

--- Data Info after cleaning 'dirty' numerics ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Month                     100000 non-null  object 
 1   Age                       95061 non-null   float64
 2   Occupation                100000 non-null  object 
 3   Annual_Income             93020 non-null   float64
 4   Monthly_Inhand_Salary     84998 non-null   float64
 5   Num_Bank_Accounts         100000 non-null  int64  
 6   Num_Credit_Card           100000 non-null  int64  
 7   Interest_Rate             100000 non-null  int64  
 8   Num_of_Loan               95215 non-null   float64
 9   Type_of_Loan              88592 non-null   object 
 10  Delay_from_due_date       100000 non-null  int64  
 11  Num_

In [4]:
# 2.3: Correcting Invalid Values and Imputing (CORRECTED SYNTAX)

# Defining an invalid age as anything less than 18 or over 100.
print(f"Rows with potentially invalid Age: {df[(df['Age'] < 18) | (df['Age'] > 100)].shape[0]}")

# Replace these invalid ages with NaN to mark them as missing.
df.loc[(df['Age'] < 18) | (df['Age'] > 100), 'Age'] = np.nan
print("Invalid ages replaced with NaN.")

# --- Create a dictionary of values to impute ---
imputation_dict = {}
cols_to_impute_with_median = [
    'Age',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Num_of_Loan',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Num_Credit_Inquiries',
    'Outstanding_Debt',
    'Amount_invested_monthly',
    'Monthly_Balance'
]

# Populate the dictionary with the median value for each column
for col in cols_to_impute_with_median:
    median_val = df[col].median()
    imputation_dict[col] = median_val

print("\nImputation dictionary created:")
print(imputation_dict)

# we use the .fillna() command with the dictionary.
df.fillna(value=imputation_dict, inplace=True)

print("\nImputation complete using the correct syntax.")
print("\n--- Data Info after correcting and imputing ---")
df.info()

Rows with potentially invalid Age: 8124
Invalid ages replaced with NaN.

Imputation dictionary created:
{'Age': np.float64(34.0), 'Annual_Income': np.float64(37550.74), 'Monthly_Inhand_Salary': np.float64(3093.745000000001), 'Num_of_Loan': np.float64(3.0), 'Num_of_Delayed_Payment': np.float64(14.0), 'Changed_Credit_Limit': np.float64(9.4), 'Num_Credit_Inquiries': np.float64(6.0), 'Outstanding_Debt': np.float64(1166.37), 'Amount_invested_monthly': np.float64(128.95453805190283), 'Monthly_Balance': np.float64(336.73122455696387)}

Imputation complete using the correct syntax.

--- Data Info after correcting and imputing ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Month                     100000 non-null  object 
 1   Age                       100000 non-null  float64
 2   Occupation                10000

In [5]:
# 2.4: Converting 'Credit_History_Age' to a Numerical Format

print("Converting 'Credit_History_Age' to total months...")

# handle any potential missing values in the column by filling them with a placeholder.
df.fillna({'Credit_History_Age':'0 Years and 0 Months'}, inplace=True)

# .str.extract(r'(\d+)') finds the first group of digits in the string.
years = df['Credit_History_Age'].str.extract(r'(\d+)').astype(int)

# This finds the second group of digits.
months = df['Credit_History_Age'].str.extract(r'(\d+)\s*Months').astype(int)

# We create a new, clean column with the result.
df['Credit_History_Age_Months'] = (years * 12) + months

# drop the original text column
df.drop(columns=['Credit_History_Age'], inplace=True)

print("Successfully created and cleaned 'Credit_History_Age_Months'.")
print("\n--- Data Info after handling Credit History Age ---")
df.info()

Converting 'Credit_History_Age' to total months...
Successfully created and cleaned 'Credit_History_Age_Months'.

--- Data Info after handling Credit History Age ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Month                      100000 non-null  object 
 1   Age                        100000 non-null  float64
 2   Occupation                 100000 non-null  object 
 3   Annual_Income              100000 non-null  float64
 4   Monthly_Inhand_Salary      100000 non-null  float64
 5   Num_Bank_Accounts          100000 non-null  int64  
 6   Num_Credit_Card            100000 non-null  int64  
 7   Interest_Rate              100000 non-null  int64  
 8   Num_of_Loan                100000 non-null  float64
 9   Type_of_Loan               88592 non-null   object 
 10  Delay_from_due_date        100000 n

In [6]:
# 2.5: Advanced Categorical Handling for 'Type_of_Loan'

from sklearn.feature_extraction.text import CountVectorizer
print("Running the 'Type_of_Loan' handling...")

# re-run the fillna for 'Type_of_Loan' if it still exists
if 'Type_of_Loan' in df.columns:
    df['Type_of_Loan'].fillna('No_Loan', inplace=True)

    # Define a smarter tokenizer function
    def smart_tokenizer(s):
        # First, replace ', and ' with a comma to standardize the separator
        s_cleaned = s.replace(', and ', ',')
        # Then, split by comma and strip whitespace from each item
        tokens = [token.strip() for token in s_cleaned.split(',')]
        return tokens

    # Use CountVectorizer with our , smarter tokenizer
    vectorizer = CountVectorizer(tokenizer=smart_tokenizer)
    loan_features = vectorizer.fit_transform(df['Type_of_Loan'])

    # Create a new DataFrame with these new, correct features
    loan_df = pd.DataFrame(loan_features.toarray(), columns=vectorizer.get_feature_names_out())
    loan_df = loan_df.add_prefix('loan_')

    print(f"Correctly created {loan_df.shape[1]} new features from 'Type_of_Loan'.")

    # Add the new features back and drop the old column
    df.reset_index(drop=True, inplace=True)
    loan_df.reset_index(drop=True, inplace=True)
    df = pd.concat([df, loan_df], axis=1)
    df.drop(columns=['Type_of_Loan'], inplace=True)

    print("\n--- Data Info after CORRECTED handling of Type_of_Loan ---")
    df.info()
else:
    print("The 'Type_of_Loan' column was already processed.")

Running the 'Type_of_Loan' handling with your improved logic...


C:\Users\Mayank\AppData\Local\Temp\ipykernel_23732\4203573149.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Type_of_Loan'].fillna('No_Loan', inplace=True)
C:\Users\Mayank\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Correctly created 11 new features from 'Type_of_Loan'.

--- Data Info after CORRECTED handling of Type_of_Loan ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 34 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Month                         100000 non-null  object 
 1   Age                           100000 non-null  float64
 2   Occupation                    100000 non-null  object 
 3   Annual_Income                 100000 non-null  float64
 4   Monthly_Inhand_Salary         100000 non-null  float64
 5   Num_Bank_Accounts             100000 non-null  int64  
 6   Num_Credit_Card               100000 non-null  int64  
 7   Interest_Rate                 100000 non-null  int64  
 8   Num_of_Loan                   100000 non-null  float64
 9   Delay_from_due_date           100000 non-null  int64  
 10  Num_of_Delayed_Payment        100000 non-null  flo

In [8]:
# 2.6: Final Encoding for Remaining Categorical and Target Variables

print("Starting final encoding step...")

# --- Part A: Encoding the Target Variable ('Credit_Score') ---
# We manually map the text labels to numbers. This is Label Encoding.
score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
df['Credit_Score'] = df['Credit_Score'].map(score_mapping)
print("Target variable 'Credit_Score' has been encoded.")

# --- Part B: Encoding the Remaining Feature Variables ---
# We use One-Hot Encoding for the rest of the object columns.
cols_to_one_hot_encode = [
    'Month',
    'Occupation',
    'Credit_Mix',
    'Payment_of_Min_Amount',
    'Payment_Behaviour'
]

# Using pd.get_dummies to convert all of them at once.
# drop_first=True helps avoid creating redundant columns.
df = pd.get_dummies(df, columns=cols_to_one_hot_encode, drop_first=True)
print("Remaining categorical features have been One-Hot Encoded.")


# --- FINAL CHECK ---
print("\n--- PREPROCESSING COMPLETE! ---")
print("Let's check our data one last time. All columns should be numerical.")
df.info()

Starting final encoding step...
Target variable 'Credit_Score' has been encoded.
Remaining categorical features have been One-Hot Encoded.

--- PREPROCESSING COMPLETE! ---
Let's check our data one last time. All columns should be numerical.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 61 columns):
 #   Column                                              Non-Null Count   Dtype  
---  ------                                              --------------   -----  
 0   Age                                                 100000 non-null  float64
 1   Annual_Income                                       100000 non-null  float64
 2   Monthly_Inhand_Salary                               100000 non-null  float64
 3   Num_Bank_Accounts                                   100000 non-null  int64  
 4   Num_Credit_Card                                     100000 non-null  int64  
 5   Interest_Rate                                       100000 non-null  i

In [9]:
# --- Phase 3: Model Training ---
# Step 3.1: Define Final Training Features and Target

# The 'df' DataFrame is our fully preprocessed training data.
# 'Credit_Score' is our target variable.
X_train_final = df.drop(columns=['Credit_Score'])
y_train_final = df['Credit_Score']

print("Final Training Features (X) and Target (y) created.")
print("Shape of X_train_final:", X_train_final.shape)
print("Shape of y_train_final:", y_train_final.shape)

Final Training Features (X) and Target (y) created.
Shape of X_train_final: (100000, 60)
Shape of y_train_final: (100000,)


In [10]:
# --- Phase 3 (Revised): Creating a Validation Set for Self-Evaluation ---


from sklearn.model_selection import train_test_split

# use our final processed features (X_train_final) and labels (y_train_final)
# split them into a training portion and a validation (testing) portion.
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_final, y_train_final, test_size=0.2, random_state=42, stratify=y_train_final)

# 'stratify=y_train_final' is very important for classification. It ensures that the
# proportion of 'Good', 'Standard', and 'Poor' scores is the same in both the
# training and validation sets, making our test more realistic.

print("Created a validation set from the main training data.")
print("New training set for validation shape:", X_train_split.shape)
print("Validation set shape:", X_val.shape)

Created a validation set from the main training data.
New training set for validation shape: (80000, 60)
Validation set shape: (20000, 60)


In [11]:
# --- Scaling, Training, and Evaluating on our Validation Set ---

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Scale the data 
scaler = StandardScaler()
X_train_split_scaled = scaler.fit_transform(X_train_split)
X_val_scaled = scaler.transform(X_val)

# 2. Train the model on the 80% split
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
print("\nTraining model on 80% of the data for validation...")
model.fit(X_train_split_scaled, y_train_split)

# 3. Evaluate on the 20% validation set (we have the answers for this!)
print("\nMaking predictions on our validation set...")
val_predictions = model.predict(X_val_scaled)

# 4. Print the performance report
print("\n--- VALIDATION PERFORMANCE ---")
print(f"Accuracy Score: {accuracy_score(y_val, val_predictions):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_predictions, target_names=['Poor (0)', 'Standard (1)', 'Good (2)']))


Training model on 80% of the data for validation...

Making predictions on our validation set...

--- VALIDATION PERFORMANCE ---
Accuracy Score: 0.7883

Classification Report:
              precision    recall  f1-score   support

    Poor (0)       0.79      0.80      0.79      5799
Standard (1)       0.81      0.81      0.81     10635
    Good (2)       0.72      0.71      0.72      3566

    accuracy                           0.79     20000
   macro avg       0.77      0.77      0.77     20000
weighted avg       0.79      0.79      0.79     20000

